# CiteFocus Lexical Search Profiling

This notebook measures where lexical retrieval time is spent.

It separates:

- SQL execution time
- row fetch / row materialization time
- Python scoring time
- merge / rerank time
- per-database total time

The goal is to answer:

- Is SQLite query execution the bottleneck?
- Are too many candidate rows being returned?
- Is Python postprocessing the bottleneck?

## 1. Setup

In [39]:
import json
import re
import sqlite3
import time
from pathlib import Path
from pprint import pprint
from IPython.display import display
import pandas as pd

ROOT = Path('/home/sascha/refcheck/CiteFocus')
PARSED_PATH = ROOT / 'outputs' / 'parsed_citations_gv_20260401_160815.json'
ARXIV_DB = ROOT / 'academic_metadata' / 'arxiv' / 'oai_pages' / 'arxiv_local_index.sqlite'
DBLP_DB = ROOT / 'academic_metadata' / 'dblp' / 'dblp_local_index.sqlite'
OPENALEX_DB = ROOT / 'academic_metadata' / 'openalex' / 'openalex_local_index.sqlite'

parsed = json.loads(PARSED_PATH.read_text())
citation = next(r for r in parsed if r['citation_id'] == 9)
pprint(citation)

{'citation_id': 9,
 'contexts': [{'expanded_context': ['A natural mitigation strategy is to '
                                    'fine-tune LLMs on security- focused '
                                    'datasets.',
                                    'However, full-parameter fine-tuning is '
                                    'computationally expensive and often '
                                    'fragile, with security gains frequently '
                                    'accompanied by catastrophic forgetting or '
                                    'degraded general coding performance '
                                    '[9–11].',
                                    'Parameter- efficient methods such as LoRA '
                                    '[12] reduce training cost but operate at '
                                    'coarse granularity, offering limited '
                                    'interpretability or control over how '
                                  

## 2. Rebuild the minimal lexical helpers locally

This notebook avoids hiding the work behind the agent function, so the timing stays interpretable.

In [40]:
STOP_WORDS = {'a', 'an', 'the', 'of', 'and', 'or', 'for', 'to', 'in', 'on', 'with', 'by'}
WORD_RE = re.compile(r"[a-zA-Z0-9]+(?:['\-][a-zA-Z0-9]+)*[?!]?")
SURNAME_PREFIXES = {'van', 'von', 'de', 'del', 'della', 'di', 'da', 'al', 'el', 'la', 'le', 'ben', 'ibn', 'mac', 'mc', 'o'}
NAME_SUFFIXES = {'jr', 'sr', 'ii', 'iii', 'iv', 'v'}
SQL_FETCH_MULTIPLIER = 2
MIN_TITLE_OVERLAP_SCORE = 0.60
WEAK_TITLE_SCORE = 0.75
MIN_MATCHED_WORDS = 2

def normalize_space(text):
    return re.sub(r'\s+', ' ', str(text or '')).strip()

def normalize_title(text):
    return re.sub(r'[^a-z0-9]+', '', str(text or '').lower())

def normalize_doi(doi):
    if not doi:
        return ''
    doi = re.sub(r'^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)', '', str(doi).strip(), flags=re.IGNORECASE)
    return doi.rstrip('.,;:').lower()

def get_query_words(text, n=8):
    text = re.sub(r'[{}]', '', str(text or ''))
    all_words = WORD_RE.findall(text)
    def is_significant(word):
        base = word.rstrip('?!')
        if base.lower() in STOP_WORDS:
            return False
        if len(base) >= 3:
            return True
        has_letter = any(c.isalpha() for c in base)
        has_digit = any(c.isdigit() for c in base)
        return has_letter and has_digit
    significant = [word for word in all_words if is_significant(word)]
    return [word.lower() for word in (significant[:n] if len(significant) >= 3 else all_words[:n])]

def build_query_words(citation):
    title = normalize_space(citation.get('parsed_title'))
    if title:
        return get_query_words(title, 8)
    raw = normalize_space(citation.get('raw_citation'))
    return get_query_words(raw, 8) if raw else []

def split_authors(text):
    return [normalize_space(x) for x in str(text or '').split(';') if normalize_space(x)]

def parse_authors_loose(text):
    normalized = normalize_space(text)
    if not normalized:
        return []
    if ';' in normalized:
        return split_authors(normalized)
    if ' and ' in normalized.lower():
        normalized = re.sub(r'\s+(and|&)\s+', ';', normalized, flags=re.IGNORECASE)
        return [normalize_space(x) for x in normalized.split(';') if normalize_space(x)]
    return [normalize_space(x) for x in normalized.split(',') if normalize_space(x)]

def get_surname_from_parts(parts):
    while len(parts) >= 2 and parts[-1].lower().rstrip('.') in NAME_SUFFIXES:
        parts = parts[:-1]
    if len(parts) >= 3 and parts[-3].lower().rstrip('.') in SURNAME_PREFIXES:
        return ' '.join(parts[-3:])
    if len(parts) >= 2 and parts[-2].lower().rstrip('.') in SURNAME_PREFIXES:
        return ' '.join(parts[-2:])
    return parts[-1]

def normalize_author(name):
    name = normalize_space(name)
    if not name:
        return ''
    if ',' in name:
        parts = name.split(',', 1)
        surname = parts[0].strip().lower()
        initials = parts[1].strip() if len(parts) > 1 else ''
        first_initial = initials[0].lower() if initials else ''
        return f'{first_initial} {surname}'.strip()
    parts = [p for p in name.split() if p]
    surname = get_surname_from_parts(parts).lower()
    return f"{parts[0][0].lower()} {surname}".strip()

def author_overlap_score(ref_authors, cand_authors):
    ref_set = {normalize_author(a) for a in ref_authors if normalize_author(a)}
    cand_set = {normalize_author(a) for a in cand_authors if normalize_author(a)}
    if not ref_set or not cand_set:
        return 0.0
    return len(ref_set & cand_set) / max(1, len(ref_set))

def title_overlap_score(query_words, candidate_title):
    query_set = {w.lower() for w in query_words if w}
    candidate_set = set(get_query_words(candidate_title, 12))
    if not query_set or not candidate_set:
        return 0.0
    return len(query_set & candidate_set) / max(1, len(query_set))

def year_support_score(parsed_year, candidate_year):
    if parsed_year is None or candidate_year is None:
        return 0.5
    parsed_year = int(parsed_year)
    candidate_year = int(candidate_year)
    if parsed_year == candidate_year:
        return 1.0
    if abs(parsed_year - candidate_year) <= 1:
        return 0.7
    if abs(parsed_year - candidate_year) <= 2:
        return 0.3
    return 0.0

def compute_title_overlap_score(query_words, candidate_title_words):
    query_set = {w.lower() for w in query_words if w}
    candidate_set = {w.lower() for w in candidate_title_words if w}
    if not query_set or not candidate_set:
        return 0.0
    return len(query_set & candidate_set) / max(1, len(query_set))

def compute_author_overlap_score(ref_authors, cand_authors):
    return author_overlap_score(ref_authors, cand_authors)

def compute_year_score(parsed_year, candidate_year):
    return year_support_score(parsed_year, candidate_year)

def sqlite_connect(path):
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    return conn

query_words = build_query_words(citation)
query_words

['large',
 'language',
 'models',
 'code',
 'security',
 'hardening',
 'adversarial',
 'testing']

## 3. Open connections

In [41]:
arxiv_conn = sqlite_connect(ARXIV_DB)
dblp_conn = sqlite_connect(DBLP_DB)
openalex_conn = sqlite_connect(OPENALEX_DB)
print('DB connections open')

DB connections open


## 4. Profiling function

This function measures the internal stages for one database:

- SQL build + execute
- fetchall
- Python scoring
- per-database total

In [42]:
def profile_single_db(citation, connection, db_name, limit=2):
    t0 = time.perf_counter()

    query_words = build_query_words(citation)
    parsed_year = citation.get('parsed_year')
    parsed_authors = citation.get('parsed_authors') or []
    parsed_title_normalized = normalize_title(citation.get('parsed_title'))
    parsed_has_identifier = bool(normalize_doi(citation.get('parsed_doi')) or citation.get('parsed_arxiv_id'))

    if db_name == 'arxiv':
        year_col = 'year'
    elif db_name == 'dblp':
        year_col = 'year'
    else:
        year_col = 'publication_year'

    placeholders = ','.join('?' for _ in query_words)
    min_word_matches = 1 if len(query_words) <= 2 else min(MIN_MATCHED_WORDS, len(query_words))

    sql = f'''
        SELECT records.*, COUNT(*) AS matched_word_count
        FROM title_words
        JOIN records ON records.id = title_words.record_id
        WHERE lower(title_words.word) IN ({placeholders})
    '''
    params = [*query_words]

    if parsed_year is not None:
        parsed_year_int = int(parsed_year)
        sql += f'''
            AND (
                {year_col} = '' OR
                {year_col} IS NULL OR
                CAST({year_col} AS INTEGER) BETWEEN ? AND ?
            )
        '''
        params.extend([parsed_year_int - 1, parsed_year_int + 1])

    sql += '''
        GROUP BY records.id
        HAVING COUNT(*) >= ?
        ORDER BY matched_word_count DESC
        LIMIT ?
    '''
    params.extend([min_word_matches, max(limit * SQL_FETCH_MULTIPLIER, limit)])

    t_sql_start = time.perf_counter()
    cursor = connection.execute(sql, params)
    t_sql_end = time.perf_counter()

    t_fetch_start = time.perf_counter()
    rows = cursor.fetchall()
    t_fetch_end = time.perf_counter()

    t_score_start = time.perf_counter()
    kept = []
    for row in rows:
        if db_name == 'arxiv':
            authors = parse_authors_loose(row['authors'])
            year = int(row['year']) if str(row['year']).isdigit() else None
            record_id = row['arxiv_id']
            url = row['url']
            doi = normalize_doi(row['doi']) or None
        elif db_name == 'dblp':
            authors = split_authors(row['authors'])
            year = int(row['year']) if str(row['year']).isdigit() else None
            record_id = row['dblp_key']
            url = row['ee']
            doi = normalize_doi(row['doi']) or None
        else:
            authors = split_authors(row['authors'])
            year = int(row['publication_year']) if str(row['publication_year']).isdigit() else None
            record_id = row['openalex_id']
            url = row['landing_page_url'] or row['openalex_id']
            doi = normalize_doi(row['doi']) or None

        title = row['title']
        title_score = title_overlap_score(query_words, title)
        if title_score < MIN_TITLE_OVERLAP_SCORE:
            continue
        author_score = author_overlap_score(parsed_authors, authors)
        year_score = year_support_score(parsed_year, year)
        normalized_title_exact = bool(parsed_title_normalized and parsed_title_normalized == normalize_title(title))
        if normalized_title_exact:
            title_score = 1.0
        lexical_score = 0.70 * title_score + 0.20 * author_score + 0.10 * year_score
        if normalized_title_exact:
            lexical_score += 0.10
        lexical_score = min(1.0, lexical_score)

        # opinionated pruning
        if title_score < 0.75 and year_score >= 0.7:
            continue
        if author_score == 0.0 and not parsed_has_identifier and title_score < 0.9:
            continue

        kept.append({
            'db': db_name,
            'record_id': record_id,
            'title': title,
            'authors': authors,
            'year': year,
            'venue': row['venue'],
            'doi': doi,
            'url': url,
            'matched_word_count': int(row['matched_word_count']),
            'title_score': round(title_score, 4),
            'author_score': round(author_score, 4),
            'year_score': round(year_score, 4),
            'lexical_score': round(lexical_score, 4),
            'normalized_title_exact': normalized_title_exact,
        })

    kept.sort(key=lambda item: (bool(item['normalized_title_exact']), item['lexical_score'], item['matched_word_count']), reverse=True)
    kept = kept[:limit]
    t_score_end = time.perf_counter()

    total_ms = round((time.perf_counter() - t0) * 1000, 2)
    return {
        'db': db_name,
        'query_words': query_words,
        'raw_row_count': len(rows),
        'kept_count': len(kept),
        'sql_time_ms': round((t_sql_end - t_sql_start) * 1000, 2),
        'fetch_time_ms': round((t_fetch_end - t_fetch_start) * 1000, 2),
        'python_scoring_time_ms': round((t_score_end - t_score_start) * 1000, 2),
        'total_db_time_ms': total_ms,
        'candidates': kept,
    }

## 5. Measure each DB separately

In [43]:
arxiv_profile = profile_single_db(citation, arxiv_conn, 'arxiv', limit=2)
dblp_profile = profile_single_db(citation, dblp_conn, 'dblp', limit=2)
openalex_profile = profile_single_db(citation, openalex_conn, 'openalex', limit=2)

for profile in [arxiv_profile, dblp_profile, openalex_profile]:
    print('\nDB:', profile['db'])
    print('  raw_row_count         =', profile['raw_row_count'])
    print('  kept_count            =', profile['kept_count'])
    print('  sql_time_ms           =', profile['sql_time_ms'])
    print('  fetch_time_ms         =', profile['fetch_time_ms'])
    print('  python_scoring_time_ms=', profile['python_scoring_time_ms'])
    print('  total_db_time_ms      =', profile['total_db_time_ms'])


DB: arxiv
  raw_row_count         = 4
  kept_count            = 1
  sql_time_ms           = 3666.76
  fetch_time_ms         = 24.78
  python_scoring_time_ms= 0.26
  total_db_time_ms      = 3691.87

DB: dblp
  raw_row_count         = 4
  kept_count            = 1
  sql_time_ms           = 11419.78
  fetch_time_ms         = 13.26
  python_scoring_time_ms= 0.39
  total_db_time_ms      = 11433.46

DB: openalex
  raw_row_count         = 4
  kept_count            = 0
  sql_time_ms           = 19649.43
  fetch_time_ms         = 0.04
  python_scoring_time_ms= 0.11
  total_db_time_ms      = 19649.61


## 6. Inspect candidates from each DB

This lets you relate time to quality and number of returned rows.

In [44]:
for profile in [arxiv_profile, dblp_profile, openalex_profile]:
    print('\n====', profile['db'].upper(), '====')
    pprint(profile['candidates'])


==== ARXIV ====
[{'author_score': 1.0,
  'authors': ['Jingxuan He', 'Martin Vechev'],
  'db': 'arxiv',
  'doi': '10.1145/3576915.3623175',
  'lexical_score': 1.0,
  'matched_word_count': 8,
  'normalized_title_exact': True,
  'record_id': '2302.05319',
  'title': 'Large Language Models for Code: Security Hardening and Adversarial '
           'Testing',
  'title_score': 1.0,
  'url': 'https://arxiv.org/abs/2302.05319',
  'venue': 'arXiv',
  'year': 2023,
  'year_score': 1.0}]

==== DBLP ====
[{'author_score': 1.0,
  'authors': ['Jingxuan He', 'Martin T. Vechev'],
  'db': 'dblp',
  'doi': '10.1145/3576915.3623175',
  'lexical_score': 1.0,
  'matched_word_count': 8,
  'normalized_title_exact': True,
  'record_id': 'conf/ccs/HeV23',
  'title': 'Large Language Models for Code: Security Hardening and Adversarial '
           'Testing.',
  'title_score': 1.0,
  'url': 'https://doi.org/10.1145/3576915.3623175',
  'venue': 'CCS',
  'year': 2023,
  'year_score': 1.0}]

==== OPENALEX ====
[]


## 7. Measure merge / rerank separately

Now we merge the already-scored candidates and time only the global merge/rerank step.

In [45]:
def merge_candidates_profiled(candidates_by_db, total_limit=2):
    t0 = time.perf_counter()
    merged = []
    seen = set()
    for db_rank, (db_name, candidates) in enumerate(candidates_by_db, start=1):
        for candidate in candidates:
            key = (candidate['db'], str(candidate['record_id']))
            if key in seen:
                continue
            seen.add(key)
            candidate = dict(candidate)
            candidate['db_priority_rank'] = db_rank
            merged.append(candidate)

    merged.sort(
        key=lambda item: (
            bool(item.get('normalized_title_exact')),
            item['lexical_score'],
            -item['db_priority_rank'],
            item['matched_word_count'],
        ),
        reverse=True,
    )
    final_candidates = merged[:total_limit]
    for idx, cand in enumerate(final_candidates, start=1):
        cand['retrieval_rank'] = idx
    t1 = time.perf_counter()
    return final_candidates, round((t1 - t0) * 1000, 2)

merged_candidates, merge_time_ms = merge_candidates_profiled([
    ('arxiv', arxiv_profile['candidates']),
    ('dblp', dblp_profile['candidates']),
    ('openalex', openalex_profile['candidates']),
], total_limit=2)

print('merge_time_ms =', merge_time_ms)
pprint(merged_candidates)

merge_time_ms = 0.02
[{'author_score': 1.0,
  'authors': ['Jingxuan He', 'Martin Vechev'],
  'db': 'arxiv',
  'db_priority_rank': 1,
  'doi': '10.1145/3576915.3623175',
  'lexical_score': 1.0,
  'matched_word_count': 8,
  'normalized_title_exact': True,
  'record_id': '2302.05319',
  'retrieval_rank': 1,
  'title': 'Large Language Models for Code: Security Hardening and Adversarial '
           'Testing',
  'title_score': 1.0,
  'url': 'https://arxiv.org/abs/2302.05319',
  'venue': 'arXiv',
  'year': 2023,
  'year_score': 1.0},
 {'author_score': 1.0,
  'authors': ['Jingxuan He', 'Martin T. Vechev'],
  'db': 'dblp',
  'db_priority_rank': 2,
  'doi': '10.1145/3576915.3623175',
  'lexical_score': 1.0,
  'matched_word_count': 8,
  'normalized_title_exact': True,
  'record_id': 'conf/ccs/HeV23',
  'retrieval_rank': 2,
  'title': 'Large Language Models for Code: Security Hardening and Adversarial '
           'Testing.',
  'title_score': 1.0,
  'url': 'https://doi.org/10.1145/3576915.3623175

## 8. Summarize bottlenecks

This cell makes it easier to see whether the problem is:

- SQL itself
- too many rows fetched
- Python scoring
- or merge/rerank

In [46]:
profiles = [arxiv_profile, dblp_profile, openalex_profile]
summary = []
for p in profiles:
    summary.append({
        'db': p['db'],
        'sql_ms': p['sql_time_ms'],
        'fetch_ms': p['fetch_time_ms'],
        'python_scoring_ms': p['python_scoring_time_ms'],
        'total_db_ms': p['total_db_time_ms'],
        'raw_rows': p['raw_row_count'],
        'kept_rows': p['kept_count'],
    })

pprint(summary)
print('\nmerge_rerank_ms =', merge_time_ms)

[{'db': 'arxiv',
  'fetch_ms': 24.78,
  'kept_rows': 1,
  'python_scoring_ms': 0.26,
  'raw_rows': 4,
  'sql_ms': 3666.76,
  'total_db_ms': 3691.87},
 {'db': 'dblp',
  'fetch_ms': 13.26,
  'kept_rows': 1,
  'python_scoring_ms': 0.39,
  'raw_rows': 4,
  'sql_ms': 11419.78,
  'total_db_ms': 11433.46},
 {'db': 'openalex',
  'fetch_ms': 0.04,
  'kept_rows': 0,
  'python_scoring_ms': 0.11,
  'raw_rows': 4,
  'sql_ms': 19649.43,
  'total_db_ms': 19649.61}]

merge_rerank_ms = 0.02


## 9. How to interpret the result

Use this rule of thumb:

- if `sql_ms` dominates: SQLite query execution is the bottleneck
- if `fetch_ms` is large and `raw_rows` is high: too many rows are being returned
- if `python_scoring_ms` dominates: the bottleneck is postprocessing/reranking
- if `merge_rerank_ms` is tiny: merge is not your problem

Repeat the profiling cell for several citations, not just one, because bottlenecks can differ by citation type.

## 10. Optimized SQL experiment

Now we test the changes you proposed in two optimized variants:

1. add useful indexes
2. precompute token document frequency in `word_stats`
3. run optimized SQL with all query words
4. run optimized SQL with only the rarest anchor words
5. aggregate on `title_words` first, join to `records` later
6. compare both optimized variants against the baseline


In [47]:
def ensure_supporting_structures(conn, year_col='year'):
    conn.execute('CREATE INDEX IF NOT EXISTS idx_title_words_word_record ON title_words(word, record_id)')
    conn.execute('CREATE INDEX IF NOT EXISTS idx_records_id ON records(id)')
    try:
        conn.execute(f'CREATE INDEX IF NOT EXISTS idx_records_year ON records({year_col})')
    except Exception:
        pass
    conn.execute('CREATE TABLE IF NOT EXISTS word_stats(word TEXT PRIMARY KEY, df INTEGER)')
    existing = conn.execute('SELECT COUNT(*) FROM word_stats').fetchone()[0]
    if existing == 0:
        conn.execute('''
            INSERT OR REPLACE INTO word_stats(word, df)
            SELECT word, COUNT(DISTINCT record_id) AS df
            FROM title_words
            GROUP BY word
        ''')
        conn.commit()

ensure_supporting_structures(arxiv_conn, 'year')
ensure_supporting_structures(dblp_conn, 'year')
ensure_supporting_structures(openalex_conn, 'publication_year')
print('Indexes and word_stats ready.')

Indexes and word_stats ready.


## 11. Choose rare anchor words instead of using all tokens

This reduces the number of matching `title_words` rows before grouping.

In [48]:
def get_token_df(conn, token):
    row = conn.execute('SELECT df FROM word_stats WHERE word = ?', (token,)).fetchone()
    return int(row[0]) if row else 10**9

def choose_anchor_tokens(query_words, conn, keep=4):
    ranked = sorted(query_words, key=lambda token: (get_token_df(conn, token), len(token)))
    return ranked[:min(keep, len(ranked))]

print('All query words:', query_words)
print('Rarest arXiv anchor words:', choose_anchor_tokens(query_words, arxiv_conn, keep=4))
print('Rarest DBLP anchor words:', choose_anchor_tokens(query_words, dblp_conn, keep=4))
print('Rarest OpenAlex anchor words:', choose_anchor_tokens(query_words, openalex_conn, keep=4))

All query words: ['large', 'language', 'models', 'code', 'security', 'hardening', 'adversarial', 'testing']
Rarest arXiv anchor words: ['hardening', 'security', 'code', 'testing']
Rarest DBLP anchor words: ['hardening', 'code', 'adversarial', 'testing']
Rarest OpenAlex anchor words: ['adversarial', 'hardening', 'code', 'testing']


## 12. Optimized profiling function

This variant keeps the same Python scoring stage so the comparison stays fair, but changes the SQL path:

- uses `title_words.word IN (...)` directly
- aggregates on `title_words` first
- joins `records` later
- can run in two modes:
  - `all`: use all query words
  - `rare`: use only a few rare anchor words


In [49]:
def profile_single_db_optimized(citation, connection, db_name, limit=2, token_strategy='rare', anchor_keep=4):
    t0 = time.perf_counter()

    all_query_words = build_query_words(citation)
    if token_strategy == 'rare':
        query_words_used = choose_anchor_tokens(all_query_words, connection, keep=anchor_keep)
    elif token_strategy == 'all':
        query_words_used = list(all_query_words)
    else:
        raise ValueError(f'Unknown token_strategy: {token_strategy}')

    parsed_year = citation.get('parsed_year')
    parsed_authors = citation.get('parsed_authors') or []
    parsed_title_normalized = normalize_title(citation.get('parsed_title'))
    parsed_has_identifier = bool(normalize_doi(citation.get('parsed_doi')) or citation.get('parsed_arxiv_id'))

    if db_name == 'arxiv':
        year_col = 'year'
    elif db_name == 'dblp':
        year_col = 'year'
    else:
        year_col = 'publication_year'

    if not query_words_used:
        return {
            'db': db_name,
            'token_strategy': token_strategy,
            'query_words_used': [],
            'raw_row_count': 0,
            'kept_count': 0,
            'sql_time_ms': 0.0,
            'fetch_time_ms': 0.0,
            'python_scoring_time_ms': 0.0,
            'total_db_time_ms': round((time.perf_counter() - t0) * 1000, 2),
            'candidates': [],
        }

    placeholders = ','.join('?' for _ in query_words_used)
    min_word_matches = 1 if len(query_words_used) <= 2 else min(2, len(query_words_used))

    sql = f'''
        SELECT r.*, tw.matched_word_count
        FROM (
            SELECT record_id, COUNT(*) AS matched_word_count
            FROM title_words
            WHERE word IN ({placeholders})
            GROUP BY record_id
            HAVING COUNT(*) >= ?
            ORDER BY matched_word_count DESC
            LIMIT ?
        ) AS tw
        JOIN records r ON r.id = tw.record_id
        WHERE (
            ? IS NULL OR r.{year_col} = '' OR r.{year_col} IS NULL OR CAST(r.{year_col} AS INTEGER) BETWEEN ? AND ?
        )
        ORDER BY tw.matched_word_count DESC
    '''
    params = [
        *query_words_used,
        min_word_matches,
        max(limit * SQL_FETCH_MULTIPLIER, limit),
        parsed_year,
        parsed_year - 1 if parsed_year else None,
        parsed_year + 1 if parsed_year else None,
    ]

    t_sql_start = time.perf_counter()
    cursor = connection.execute(sql, params)
    t_sql_end = time.perf_counter()

    t_fetch_start = time.perf_counter()
    rows = cursor.fetchall()
    t_fetch_end = time.perf_counter()

    t_score_start = time.perf_counter()
    kept = []
    for row in rows:
        if db_name == 'arxiv':
            authors = parse_authors_loose(row['authors'])
            year = int(row['year']) if str(row['year']).isdigit() else None
            record_id = row['arxiv_id']
            url = row['url']
            doi = normalize_doi(row['doi']) or None
            raw_title = row['title']
        elif db_name == 'dblp':
            authors = parse_authors_loose(row['authors'])
            year = int(row['year']) if str(row['year']).isdigit() else None
            record_id = row['dblp_key']
            url = row['ee'] or row['url']
            doi = normalize_doi(row['doi']) or None
            raw_title = row['title']
        else:
            authors = parse_authors_loose(row['authors'])
            year_raw = row['publication_year']
            year = int(year_raw) if str(year_raw).isdigit() else None
            record_id = row['openalex_id']
            url = row['landing_page_url'] or row['openalex_id']
            doi = normalize_doi(row['doi']) or None
            raw_title = row['title']

        title = normalize_space(raw_title or '')
        title_words = build_query_words({'parsed_title': title})
        candidate_title_normalized = normalize_title(title)
        normalized_title_exact = bool(
            parsed_title_normalized and candidate_title_normalized and parsed_title_normalized == candidate_title_normalized
        )

        title_score = compute_title_overlap_score(build_query_words(citation), title_words)
        if normalized_title_exact:
            title_score = max(title_score, 1.0)
        if title_score < MIN_TITLE_OVERLAP_SCORE:
            continue

        author_score = compute_author_overlap_score(parsed_authors, authors)
        year_score = compute_year_score(citation.get('parsed_year'), year)
        lexical_score = 0.65 * title_score + 0.25 * author_score + 0.10 * year_score
        if normalized_title_exact:
            lexical_score += 0.10

        if title_score < WEAK_TITLE_SCORE and year_score >= 0.7:
            continue
        if author_score == 0.0 and not parsed_has_identifier and title_score < 0.9:
            continue

        kept.append({
            'db': db_name,
            'record_id': record_id,
            'title': title,
            'authors': authors,
            'year': year,
            'venue': row['venue'] if 'venue' in row.keys() else None,
            'doi': doi,
            'url': url,
            'matched_word_count': row['matched_word_count'],
            'normalized_title_exact': normalized_title_exact,
            'title_score': round(title_score, 4),
            'author_score': round(author_score, 4),
            'year_score': round(year_score, 4),
            'lexical_score': round(lexical_score, 4),
        })

    kept.sort(
        key=lambda c: (
            c['normalized_title_exact'],
            c['lexical_score'],
            c['matched_word_count'],
        ),
        reverse=True,
    )
    kept = kept[:limit]
    t_score_end = time.perf_counter()

    total = time.perf_counter() - t0
    return {
        'db': db_name,
        'token_strategy': token_strategy,
        'query_words_used': query_words_used,
        'raw_row_count': len(rows),
        'kept_count': len(kept),
        'sql_time_ms': round((t_sql_end - t_sql_start) * 1000, 2),
        'fetch_time_ms': round((t_fetch_end - t_fetch_start) * 1000, 2),
        'python_scoring_time_ms': round((t_score_end - t_score_start) * 1000, 2),
        'total_db_time_ms': round(total * 1000, 2),
        'candidates': kept,
    }


In [50]:
optimized_profiles = []
for strategy in ['all', 'rare']:
    optimized_profiles.extend([
        profile_single_db_optimized(citation, arxiv_conn, 'arxiv', limit=2, token_strategy=strategy),
        profile_single_db_optimized(citation, dblp_conn, 'dblp', limit=2, token_strategy=strategy),
        profile_single_db_optimized(citation, openalex_conn, 'openalex', limit=2, token_strategy=strategy),
    ])

for profile in optimized_profiles:
    print(f"\nDB: {profile['db']} | strategy: {profile['token_strategy']}")
    print('  query_words_used      =', profile['query_words_used'])
    print('  raw_row_count         =', profile['raw_row_count'])
    print('  kept_count            =', profile['kept_count'])
    print('  sql_time_ms           =', profile['sql_time_ms'])
    print('  fetch_time_ms         =', profile['fetch_time_ms'])
    print('  python_scoring_time_ms=', profile['python_scoring_time_ms'])
    print('  total_db_time_ms      =', profile['total_db_time_ms'])



DB: arxiv | strategy: all
  query_words_used      = ['large', 'language', 'models', 'code', 'security', 'hardening', 'adversarial', 'testing']
  raw_row_count         = 1
  kept_count            = 1
  sql_time_ms           = 88.05
  fetch_time_ms         = 0.33
  python_scoring_time_ms= 0.12
  total_db_time_ms      = 88.57

DB: dblp | strategy: all
  query_words_used      = ['large', 'language', 'models', 'code', 'security', 'hardening', 'adversarial', 'testing']
  raw_row_count         = 2
  kept_count            = 1
  sql_time_ms           = 144.04
  fetch_time_ms         = 0.77
  python_scoring_time_ms= 0.16
  total_db_time_ms      = 145.0

DB: openalex | strategy: all
  query_words_used      = ['large', 'language', 'models', 'code', 'security', 'hardening', 'adversarial', 'testing']
  raw_row_count         = 2
  kept_count            = 0
  sql_time_ms           = 59.48
  fetch_time_ms         = 0.37
  python_scoring_time_ms= 0.08
  total_db_time_ms      = 59.95

DB: arxiv | strate

## 13. Baseline vs optimized comparison

Now compare three variants for each database:

- `baseline`: current notebook baseline
- `optimized_all`: optimized SQL shape, but still using all query words
- `optimized_rare`: optimized SQL shape with rare-anchor selection


In [51]:
baseline_profiles = {p['db']: p for p in [arxiv_profile, dblp_profile, openalex_profile]}
optimized_profile_map = {(p['db'], p['token_strategy']): p for p in optimized_profiles}

comparison = []
for db_name in ['arxiv', 'dblp', 'openalex']:
    base = baseline_profiles[db_name]
    opt_all = optimized_profile_map[(db_name, 'all')]
    opt_rare = optimized_profile_map[(db_name, 'rare')]
    comparison.append({
        'db': db_name,
        'baseline_sql_ms': base['sql_time_ms'],
        'optimized_all_sql_ms': opt_all['sql_time_ms'],
        'optimized_rare_sql_ms': opt_rare['sql_time_ms'],
        'baseline_fetch_ms': base['fetch_time_ms'],
        'optimized_all_fetch_ms': opt_all['fetch_time_ms'],
        'optimized_rare_fetch_ms': opt_rare['fetch_time_ms'],
        'baseline_python_ms': base['python_scoring_time_ms'],
        'optimized_all_python_ms': opt_all['python_scoring_time_ms'],
        'optimized_rare_python_ms': opt_rare['python_scoring_time_ms'],
        'baseline_total_ms': base['total_db_time_ms'],
        'optimized_all_total_ms': opt_all['total_db_time_ms'],
        'optimized_rare_total_ms': opt_rare['total_db_time_ms'],
        'baseline_raw_rows': base['raw_row_count'],
        'optimized_all_raw_rows': opt_all['raw_row_count'],
        'optimized_rare_raw_rows': opt_rare['raw_row_count'],
        'optimized_all_words': opt_all['query_words_used'],
        'optimized_rare_words': opt_rare['query_words_used'],
    })

comparison_df = pd.DataFrame(comparison)
display(comparison_df)

pretty_columns = [
    'db',
    'baseline_sql_ms', 'optimized_all_sql_ms', 'optimized_rare_sql_ms',
    'baseline_fetch_ms', 'optimized_all_fetch_ms', 'optimized_rare_fetch_ms',
    'baseline_python_ms', 'optimized_all_python_ms', 'optimized_rare_python_ms',
    'baseline_total_ms', 'optimized_all_total_ms', 'optimized_rare_total_ms',
    'baseline_raw_rows', 'optimized_all_raw_rows', 'optimized_rare_raw_rows',
]
display(comparison_df[pretty_columns])


,db,baseline_sql_ms,optimized_all_sql_ms,optimized_rare_sql_ms,baseline_fetch_ms,optimized_all_fetch_ms,optimized_rare_fetch_ms,baseline_python_ms,optimized_all_python_ms,optimized_rare_python_ms,baseline_total_ms,optimized_all_total_ms,optimized_rare_total_ms,baseline_raw_rows,optimized_all_raw_rows,optimized_rare_raw_rows,optimized_all_words,optimized_rare_words
0,arxiv,3666.76,88.05,4.81,24.78,0.33,0.03,0.26,0.12,0.13,3691.87,88.57,5.14,4,1,2,"[large, language, models, code, security, hard...","[hardening, security, code, testing]"
1,dblp,11419.78,144.04,25.69,13.26,0.77,0.20,0.39,0.16,0.08,11433.46,145.00,26.14,4,2,1,"[large, language, models, code, security, hard...","[hardening, code, adversarial, testing]"
2,openalex,19649.43,59.48,10.29,0.04,0.37,0.00,0.11,0.08,0.00,19649.61,59.95,10.46,4,2,0,"[large, language, models, code, security, hard...","[adversarial, hardening, code, testing]"


,db,baseline_sql_ms,optimized_all_sql_ms,optimized_rare_sql_ms,baseline_fetch_ms,optimized_all_fetch_ms,optimized_rare_fetch_ms,baseline_python_ms,optimized_all_python_ms,optimized_rare_python_ms,baseline_total_ms,optimized_all_total_ms,optimized_rare_total_ms,baseline_raw_rows,optimized_all_raw_rows,optimized_rare_raw_rows
0,arxiv,3666.76,88.05,4.81,24.78,0.33,0.03,0.26,0.12,0.13,3691.87,88.57,5.14,4,1,2
1,dblp,11419.78,144.04,25.69,13.26,0.77,0.20,0.39,0.16,0.08,11433.46,145.00,26.14,4,2,1
2,openalex,19649.43,59.48,10.29,0.04,0.37,0.00,0.11,0.08,0.00,19649.61,59.95,10.46,4,2,0


## 14. How to read the comparison

Use the three-way comparison to isolate the real win:

- if `optimized_all_sql_ms` is already much lower than baseline, the SQL shape and indexes help by themselves
- if `optimized_rare_sql_ms` drops much further than `optimized_all_sql_ms`, rare-word selection is doing the heavy lifting
- if raw row counts shrink mainly in `optimized_rare`, the broad token set was the main problem
- if Python scoring stays high even after row counts drop, postprocessing is still the bottleneck
- if there is little difference between `optimized_all` and `optimized_rare`, rare-word selection is not the main lever for this citation


## 15. Cleanup

In [52]:
arxiv_conn.close()
dblp_conn.close()
openalex_conn.close()
print('Connections closed.')

Connections closed.
